# Prepare h5ad object for label transfer
Using logistic regression

In [50]:
import warnings, sys
import anndata2ri
anndata2ri.activate() 
warnings.filterwarnings('ignore')

%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [51]:
%%R
# Load packages
suppressPackageStartupMessages({
    library(reticulate)
    library(BiocParallel)
    library(SingleCellExperiment)
    library(dplyr)
    library(Matrix)
})
    ncores = 4
    mcparam = MulticoreParam(workers = ncores)
    register(mcparam)

options(repr.plot.width=15, repr.plot.height=8)

# define directories
main = "/rds/project/rds-SDzz0CATGms/users/bt392/phd_01_Eomes_RNA/"
in_dir = paste0(main, '07_atlas_mapping/')
out_dir = paste0(main, '07_atlas_mapping/label_transfer/')
dir.create(out_dir, showWarnings = FALSE)


# load in data
big_sce = readRDS(paste0(in_dir,"big_sce.rds"))

chim_meta =  read.csv2(paste0(main, 'meta.tab'), sep='\t')

atlas_in = '/rds/project/rds-SDzz0CATGms/users/bt392/mouse/Mixl1_KO/atlas/atlas/'
atlas_meta = read.table(paste0(atlas_in, 'meta.tab'), header = TRUE, sep = "\t", stringsAsFactors = FALSE, comment.char = "$")

# match formats
chim_meta$cell = paste0('chim_', chim_meta$cell)
atlas_meta$cell = paste0('atlas_', atlas_meta$cell)
chim_meta$sample = paste0('chim_', chim_meta$sample)
atlas_meta$sample = paste0('atlas_', atlas_meta$sample)

atlas_meta = atlas_meta[!atlas_meta$doublet,]
atlas_meta = atlas_meta[!atlas_meta$stripped,]

chim_meta = chim_meta[!chim_meta$doublet,]
chim_meta = chim_meta[!chim_meta$stripped,]

chim_meta = chim_meta %>% select(cell, barcode, stage, sample, Sample_name, tdTom)
atlas_meta = atlas_meta %>% select(cell, barcode, stage, sample, celltype)

# write metadata
write.table(chim_meta, file = paste0(out_dir, "chim_meta.tsv"), sep='\t', col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(atlas_meta, file = paste0(out_dir, "atlas_meta.tsv"), sep='\t', col.names = TRUE, row.names = FALSE, quote = FALSE)

# separate atlas from experiment
chim_sce = big_sce[, chim_meta$cell]
atlas_sce = big_sce[, atlas_meta$cell]

# write sce
saveRDS(chim_sce, file = paste0(out_dir,"chim_sce.rds"))
saveRDS(atlas_sce, file = paste0(out_dir,"atlas_sce.rds"))

In [65]:
%%R
write.table(chim_meta, file = paste0(out_dir, "chim_meta.tsv"), sep='\t', col.names = TRUE, row.names = FALSE, quote = FALSE)
write.table(atlas_meta, file = paste0(out_dir, "atlas_meta.tsv"), sep='\t', col.names = TRUE, row.names = FALSE, quote = FALSE)

In [52]:
%%R
saveRDS(chim_sce, file = paste0(out_dir,"chim_sce.rds"))
saveRDS(atlas_sce, file = paste0(out_dir,"atlas_sce.rds"))

In [55]:
%%R
counts(chim_sce)[1:10,1:10]

10 x 10 sparse Matrix of class "dgCMatrix"
                            
 [1,] 3 6   . 6 6 1  2 6 5 4
 [2,] . .   . . . .  . . . .
 [3,] . 1   . 1 . .  . 1 3 3
 [4,] . 4 142 1 . 5 46 . . 3
 [5,] . .   . . . .  . . . .
 [6,] . .   . . . .  . . . .
 [7,] 2 .   2 1 1 .  1 3 1 1
 [8,] . .   . . . .  . . . .
 [9,] . .   2 1 . .  1 1 . .
[10,] . .   . . 1 .  . . 1 .


In [ ]:
%%R
colnames(counts(chim_sce)) = NULL
colnames(counts(chim_sce)) = NULL

In [57]:
%%R -o adata
adata <- readRDS('/rds/project/rds-SDzz0CATGms/users/bt392/phd_01_Eomes_RNA/07_atlas_mapping/label_transfer/chim_sce.rds')

In [59]:
%%R 
adata

class: SingleCellExperiment 
dim: 29452 27986 
metadata(0):
assays(1): X
rownames(29452): ENSMUSG00000000001 ENSMUSG00000000003 ...
  ENSMUSG00000114189 ENSMUSG00000114192
rowData names(0):
colnames(27986): chim_cell_1 chim_cell_2 ... chim_cell_30169
  chim_cell_30171
colData names(0):
reducedDimNames(0):
altExpNames(0):


In [58]:
adata

<rpy2.robjects.methods.RS4 object at 0x2b5fbdee4700> [RTYPES.S4SXP]
R classes: ('SingleCellExperiment',)

In [ ]:
%%R
# write raw count & new metadata
writeMM(counts(chim_sce), file = paste0(out_dir,"chim_counts.mtx"))
writeMM(counts(atlas_sce), file = paste0(out_dir,"atlas_counts.mtx"))

write.table(chim_meta, file = paste0(out_dir, "chim_meta.tsv"), col.names = FALSE, row.names = FALSE, quote = FALSE)
write.table(atlas_meta, file = paste0(out_dir, "atlas_meta.tsv"), col.names = FALSE, row.names = FALSE, quote = FALSE)

In [43]:
%%R -o chim_counts
chim_counts = counts(chim_sce)

In [49]:
%%R
head(chim_sce)
chim_sce@metadata = NULL
head(chim_sce)

2021-10-13 16:27:59,706 [47760] WARNING  rpy2.rinterface_lib.callbacks:123: [JupyterRequire] R[write to console]: Error in (function (cl, name, valueClass)  : 
  assignment of an object of class “NULL” is not valid for @‘metadata’ in an object of class “SingleCellExperiment”; is(value, "list") is not TRUE




Error in (function (cl, name, valueClass)  : 
  assignment of an object of class “NULL” is not valid for @‘metadata’ in an object of class “SingleCellExperiment”; is(value, "list") is not TRUE


In [46]:
def matrix(df):
  temp = pd.DataFrame.sparse.from_spmatrix(df)
  return temp

In [60]:
import anndata2ri
import scanpy as sc
import pandas as pd
import numpy as np

anndata2ri.activate()
%reload_ext rpy2.ipython

In [61]:
%%R -o chim_adata
chim_adata <- readRDS('/rds/project/rds-SDzz0CATGms/users/bt392/phd_01_Eomes_RNA/07_atlas_mapping/label_transfer/chim_sce.rds')

In [62]:
chim_adata

<rpy2.robjects.methods.RS4 object at 0x2b5fbdea8940> [RTYPES.S4SXP]
R classes: ('SingleCellExperiment',)

In [40]:
print(adata_allen) 

AttributeError: 'RS4' object has no attribute 'X'

In [30]:
%%R -o adata
adata <- readRDS('/rds/project/rds-SDzz0CATGms/users/bt392/phd_01_Eomes_RNA/07_atlas_mapping/big_sce.rds')

In [42]:
%%R -o adata_allen
adata_allen <- as(chim_sce, 'SingleCellExperiment')

2021-10-13 16:18:12,540 [47760] WARNING  rpy2.rinterface_lib.callbacks:123: [JupyterRequire] R[write to console]: Error in as(chim_sce, "anndata") : 
  no method or default for coercing “SingleCellExperiment” to “anndata”




Error in as(chim_sce, "anndata") : 
  no method or default for coercing “SingleCellExperiment” to “anndata”


In [36]:
import anndata2ri
anndata2ri.activate()
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [35]:
import scanpy as sc
import pandas as pd
import numpy as np

In [ ]:
%%R -o atlas_adata
atlas_adata <- atlas_sce

In [22]:
%%R 
chim_adata

class: SingleCellExperiment 
dim: 29452 27986 
metadata(6): cell barcode ... Sample_name tdTom
assays(1): X
rownames(29452): ENSMUSG00000000001 ENSMUSG00000000003 ...
  ENSMUSG00000114189 ENSMUSG00000114192
rowData names(0):
colnames(27986): chim_cell_1 chim_cell_2 ... chim_cell_30169
  chim_cell_30171
colData names(0):
reducedDimNames(0):
altExpNames(0):


In [20]:
%%R -o chim_adata
chim_adata <- readRDS(paste0(out_dir,"chim_sce.rds"))

In [19]:
anndata2ri.activate()
%reload_ext rpy2.ipython

In [12]:
%%R 
chim_sce@metadata = chim_meta
atlas_sce@metadata = atlas_meta

saveRDS(chim_sce, file = paste0(out_dir,"chim_sce.rds"))
saveRDS(atlas_sce, file = paste0(out_dir,"atlas_sce.rds"))

In [44]:
chim_counts

<rpy2.robjects.methods.RS4 object at 0x2b5fbee10c00> [RTYPES.S4SXP]
R classes: ('dgCMatrix',)

In [47]:
test = matrix(chim_counts)

AttributeError: 'RS4' object has no attribute 'tocsc'

In [23]:
chim_adata.X

AttributeError: 'RS4' object has no attribute 'X'

In [29]:
%reload_ext rpy2.ipython

In [34]:
anndata2ri.activate()

In [9]:
main = '/rds/project/rds-SDzz0CATGms/users/bt392/mouse/Mixl1_KO/atlas/atlas/'
chim_counts = 'chim_counts.mtx'
atlas_counts = 'atlas_counts.mtx'

chim_meta = 'chim_meta.tsv'
atlas_meta = 'atlas_meta.tsv'